# Real-stream discovery atlas (fine-tuned CPEN)

Score **every** blob in `galaxies_real/train/0000` with a fine-tuned cell-split checkpoint, then show:

| Role | Meaning |
|------|---------|
| **Train cells** | Supervised at fine-tune (contain train-stream members) |
| **Val cells** | Held out of train cells for threshold / model selection |
| **Test / discovery cells** | Everything else — the sky we claim to *discover* on |

For each star at a probability cut \(t^\star\):

- **TP** — predicted stream & catalogued S5 (recovered)
- **FN** — catalogued S5 missed
- **FP** — predicted stream but field / uncatalogued

Default holdout streams (never in the train-stream set): `Elqui, Chenab, Indus, Phoenix, AAU`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.lines import Line2D

REPO = Path.cwd().resolve()
for cand in (REPO, *REPO.parents):
    if (cand / "src" / "cpen").is_dir():
        REPO = cand
        break
sys.path[:0] = [str(REPO), str(REPO / "src")]

from cpen.apps.streams.cell_stream_split import describe_cell_split
from cpen.apps.streams.preprocess.config import DEFAULT_REAL_PIPELINE_ROOT
from cpen.apps.streams.real_discovery import (
    DiscoveryConfig,
    apply_threshold,
    cell_role,
    choose_threshold_on_val,
    fit_stats_for_cfg,
    galaxy_dir,
    load_lit_from_run,
    rebuild_cell_split,
    score_manifest,
    stream_recovery_table,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("repo", REPO)
print("device", DEVICE)


## 0. Point at a fine-tune run

Edit `RUN_DIR` if you want a different checkpoint (e.g. the lower-LR job once it finishes).


In [ ]:
# Default: η₀=0.01 cellsplit fine-tune (update when 13880706 finishes)
RUN_DIR = Path(
    "/scratch/gpfs/BHANIN/jdezoort/cpen_runs/blobs_real_ft/stream_output/adamw/sweep_lr/"
    "cpen_4_256_0p01_t10_ln_cosine_op-incidence_onorm-degree_res-transformer-like_drop0p1_nf6_gnativeedges_"
    "blobs-rich10-dropparallax-hyper-cellsplit-g0000-initfrom-edgecons"
)

CFG = DiscoveryConfig(
    real_root=Path(DEFAULT_REAL_PIPELINE_ROOT),
    galaxy_id="0000",
    drop_features="parallax",
    rich_min_mock=10,
    cell_val_frac=0.15,
    split_seed=0,
)

assert RUN_DIR.is_dir(), RUN_DIR
print("run_dir", RUN_DIR)
print("galaxy", galaxy_dir(CFG))


## 1. Rebuild cell split + load model


In [ ]:
CELL_SPLIT = rebuild_cell_split(CFG)
print(describe_cell_split(CELL_SPLIT))
print(
    f"cells train/val/test = "
    f"{len(CELL_SPLIT.train_cells)}/{len(CELL_SPLIT.val_cells)}/{len(CELL_SPLIT.test_cells)}"
)
print("holdout streams:", ", ".join(sorted(CELL_SPLIT.holdout_streams)))

STATS, _split2, train_man = fit_stats_for_cfg(CFG)
assert set(_split2.train_cells) == set(CELL_SPLIT.train_cells)
print(f"feature stats from {len(train_man):,} train-pool graphs")

LIT, CKPT, HP, DEVICE = load_lit_from_run(RUN_DIR, cfg=CFG, device=DEVICE)
print("ckpt", CKPT)
print("hparams", HP)


## 2. Score every blob in the galaxy

This is the slow step (~minutes on one GPU). Results stay in memory as `SCORED`.


In [ ]:
# Set MAX_GRAPHS to a small int for a dry run; None = all blobs with n_stars >= min_stars
# Full atlas (~all cells) needs GPU; on CPU prefer MAX_GRAPHS=50 first.
MAX_GRAPHS = None

SCORED = score_manifest(
    CFG,
    lit=LIT,
    stats=STATS,
    cell_split=CELL_SPLIT,
    device=DEVICE,
    max_graphs=MAX_GRAPHS,
    progress=True,
)
print(f"scored {len(SCORED):,} blobs")
print(pd.Series([r['role'] for r in SCORED]).value_counts())


## 3. Choose \(t^\star\) on **val cells only** (no peeking at discovery)


In [ ]:
T_STAR, SWEEP = choose_threshold_on_val(SCORED)
print(f"t* (max F1 on val cells) = {T_STAR:.3f}")
display(SWEEP.sort_values('f1', ascending=False).head(5))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(SWEEP['t'], SWEEP['f1'], label='F1', lw=2)
ax.plot(SWEEP['t'], SWEEP['prec'], label='precision', alpha=0.8)
ax.plot(SWEEP['t'], SWEEP['rec'], label='recall', alpha=0.8)
ax.axvline(T_STAR, color='k', ls='--', lw=1, label=f't*={T_STAR:.2f}')
ax.set_xlabel('threshold'); ax.set_ylabel('score')
ax.set_title('Threshold sweep on val cells only')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

STARS = apply_threshold(SCORED, T_STAR)
print(f"stars total={len(STARS):,}  pred+={(STARS.pred).sum():,}  S5+={STARS.y.sum():,}")


## 4. Galaxy overview — train vs discovery sky

Left: cell roles. Right: catalogued S5 members colored by stream, with discovery-cell FPs/TPs overlaid.


In [ ]:
def _cell_centers():
    # Prefer DES footprint centers if available; else mean RA/Dec per cell from stars.
    rows = []
    for cid, sub in STARS.groupby('cell_id'):
        rows.append(dict(cell_id=int(cid), ra=float(sub.ra.mean()), dec=float(sub.dec.mean()),
                         role=cell_role(int(cid), CELL_SPLIT)))
    return pd.DataFrame(rows)

CENTERS = _cell_centers()
ROLE_COLOR = {'train': '#1b9e77', 'val': '#d95f02', 'test': '#7570b3'}

fig, axs = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

# --- left: cell roles ---
ax = axs[0]
# faint field
field = STARS[~STARS.y]
ax.scatter(field.ra.values[::20], field.dec.values[::20], s=1, c='#dddddd', alpha=0.4, rasterized=True)
for role, color in ROLE_COLOR.items():
    m = CENTERS.role == role
    ax.scatter(CENTERS.loc[m, 'ra'], CENTERS.loc[m, 'dec'], s=80, c=color, label=f'{role} cell',
               edgecolors='k', linewidths=0.4, zorder=3)
ax.set_title('Cell roles (fine-tune split)')
ax.set_xlabel('RA [deg]'); ax.set_ylabel('Dec [deg]')
ax.legend(frameon=False, loc='best')
ax.invert_xaxis()

# --- right: S5 truth + discovery predictions ---
ax = axs[1]
ax.scatter(field.ra.values[::20], field.dec.values[::20], s=1, c='#eeeeee', alpha=0.5, rasterized=True)
s5 = STARS[STARS.y]
# color by stream
streams = sorted(s5.stream_label.unique())
cmap = plt.cm.tab20(np.linspace(0, 1, max(len(streams), 1)))
for i, name in enumerate(streams):
    m = s5.stream_label == name
    ax.scatter(s5.loc[m, 'ra'], s5.loc[m, 'dec'], s=4, c=[cmap[i % len(cmap)]],
               label=name if i < 12 else None, alpha=0.7, rasterized=True)
# discovery-cell TPs / FPs
disc = STARS[STARS.role == 'test']
ax.scatter(disc.loc[disc.tp, 'ra'], disc.loc[disc.tp, 'dec'], s=18, facecolors='none',
           edgecolors='lime', linewidths=0.6, label='discovery TP', zorder=4)
ax.scatter(disc.loc[disc.fp, 'ra'], disc.loc[disc.fp, 'dec'], s=14, facecolors='none',
           edgecolors='cyan', linewidths=0.5, label='discovery FP', zorder=4)
ax.set_title(f'Catalog S5 + discovery preds @ t*={T_STAR:.2f}')
ax.set_xlabel('RA [deg]')
ax.invert_xaxis()
ax.legend(frameon=False, loc='upper left', fontsize=8, markerscale=2, ncol=2)
plt.tight_layout()
plt.show()


## 4b. Full-sky cell mosaic — trained / discovered / missed

One panel per cell (RA–Dec). Color encodes triage at $t^\star$:

- lime = recovered S5 (TP)
- crimson = missed S5 (FN)
- cyan outline = predicted stream, not in catalog (FP)
- gray = field TN

Panel border: **green** = train, **orange** = val, **purple** = discovery (test).


In [ ]:
def plot_cell_mosaic(stars: pd.DataFrame, *, roles=None, max_cells: int = 112, ncols: int = 8):
    """Compact atlas of every cell — the whole-galaxy discovery map."""
    tab = stars if roles is None else stars[stars.role.isin(list(roles))]
    cells = (
        tab.groupby(['cell_id', 'role'], as_index=False)
        .agg(n_s5=('y', 'sum'), n_tp=('tp', 'sum'), n_fn=('fn', 'sum'), mean_p=('p', 'mean'))
        .sort_values(['role', 'n_s5', 'mean_p'], ascending=[True, False, False])
        .head(max_cells)
    )
    n = len(cells)
    if n == 0:
        print('no cells to mosaic')
        return
    nrows = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * 1.55, nrows * 1.45),
                            sharex=False, sharey=False)
    axs = np.atleast_2d(axs)
    border = {'train': '#1b9e77', 'val': '#d95f02', 'test': '#7570b3'}
    for i, row in enumerate(cells.itertuples(index=False)):
        ax = axs[i // ncols, i % ncols]
        sub = tab[tab.cell_id == row.cell_id]
        ax.scatter(sub.loc[sub.tn, 'ra'], sub.loc[sub.tn, 'dec'], s=0.4, c='#e8e8e8', rasterized=True)
        ax.scatter(sub.loc[sub.tp, 'ra'], sub.loc[sub.tp, 'dec'], s=2.5, c='lime', rasterized=True)
        ax.scatter(sub.loc[sub.fn, 'ra'], sub.loc[sub.fn, 'dec'], s=2.5, c='crimson', rasterized=True)
        ax.scatter(sub.loc[sub.fp, 'ra'], sub.loc[sub.fp, 'dec'], s=2.0, facecolors='none',
                   edgecolors='cyan', linewidths=0.3, rasterized=True)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_color(border.get(row.role, 'k'))
            spine.set_linewidth(2.0)
        tag = f"{int(row.cell_id)}"
        if row.n_s5:
            tag += f" S5={int(row.n_s5)}"
        ax.set_title(tag, fontsize=7, pad=1)
        ax.invert_xaxis()
    for j in range(n, nrows * ncols):
        axs[j // ncols, j % ncols].axis('off')
    fig.suptitle(f'Cell mosaic @ t*={T_STAR:.2f}  (border = role)', fontsize=12, y=1.01)
    handles = [
        Line2D([0], [0], color='#1b9e77', lw=3, label='train'),
        Line2D([0], [0], color='#d95f02', lw=3, label='val'),
        Line2D([0], [0], color='#7570b3', lw=3, label='discovery'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='lime', markersize=6, label='TP'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='crimson', markersize=6, label='FN'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='none', markeredgecolor='cyan', markersize=6, label='FP'),
    ]
    fig.legend(handles=handles, loc='upper center', ncol=6, frameon=False, fontsize=8,
               bbox_to_anchor=(0.5, 0.0))
    plt.tight_layout()
    plt.show()

plot_cell_mosaic(STARS, max_cells=112, ncols=8)


## 5. What was trained / discovered / missed (by stream)

Recall on **test cells** is the discovery metric. Train-cell recall is a sanity check (model should still fire there).


In [ ]:
rec_test = stream_recovery_table(STARS, roles=['test'])
rec_train = stream_recovery_table(STARS, roles=['train', 'val'])
rec_all = stream_recovery_table(STARS)

holdout = set(CELL_SPLIT.holdout_streams)
rec_test['is_holdout'] = rec_test.stream_label.isin(holdout)
rec_train['is_holdout'] = rec_train.stream_label.isin(holdout)

print('=== Discovery cells (test) ===')
display(rec_test)
print('=== Supervised cells (train+val) ===')
display(rec_train)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
for ax, tab, title in (
    (axs[0], rec_test, 'Recall in discovery (test) cells'),
    (axs[1], rec_train, 'Recall in supervised (train+val) cells'),
):
    if not len(tab):
        ax.set_visible(False)
        continue
    colors = ['#d95f02' if h else '#1b9e77' for h in tab.is_holdout]
    ax.barh(tab.stream_label[::-1], tab.recall[::-1], color=colors[::-1])
    ax.set_xlim(0, 1)
    ax.set_xlabel('recall @ t*')
    ax.set_title(title)
    ax.axvline(0.5, color='k', ls=':', lw=0.8)
handles = [
    Line2D([0], [0], color='#d95f02', lw=6, label='holdout stream'),
    Line2D([0], [0], color='#1b9e77', lw=6, label='train stream'),
]
axs[1].legend(handles=handles, frameon=False, loc='lower right')
plt.tight_layout()
plt.show()


## 6. Per-cell atlas — truth | score | discovery triage

For each cell: catalog labels, model \(p\), and TP/FP/FN at \(t^\star\).  
Discovery cells are listed first (the interesting ones).


In [ ]:
def plot_cell_discovery(cell_id: int, stars: pd.DataFrame = STARS, t: float = T_STAR):
    sub = stars[stars.cell_id == int(cell_id)]
    if not len(sub):
        print(f'cell {cell_id}: no scored stars')
        return
    role = cell_role(int(cell_id), CELL_SPLIT)
    fig, axs = plt.subplots(1, 3, figsize=(13.5, 4.2), sharex=True, sharey=True)

    # field backdrop
    for ax in axs:
        f = sub[~sub.y]
        ax.scatter(f.ra, f.dec, s=2, c='#e8e8e8', rasterized=True)
        ax.invert_xaxis()
        ax.set_xlabel('RA')

    # (0) catalog
    ax = axs[0]
    s5 = sub[sub.y]
    for name, g in s5.groupby('stream_label'):
        ax.scatter(g.ra, g.dec, s=8, label=name, alpha=0.85)
    ax.set_title(f'cell {cell_id} [{role}]  catalog')
    ax.set_ylabel('Dec')
    if len(s5):
        ax.legend(frameon=False, fontsize=7, markerscale=1.4, loc='best')

    # (1) score
    ax = axs[1]
    sc = ax.scatter(sub.ra, sub.dec, s=6, c=sub.p, cmap='magma', vmin=0, vmax=1, rasterized=True)
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02, label='p(stream)')
    ax.set_title('model score')

    # (2) triage
    ax = axs[2]
    ax.scatter(sub.loc[sub.tn, 'ra'], sub.loc[sub.tn, 'dec'], s=2, c='#eeeeee', rasterized=True)
    ax.scatter(sub.loc[sub.tp, 'ra'], sub.loc[sub.tp, 'dec'], s=10, c='lime', label=f'TP={int(sub.tp.sum())}', alpha=0.9)
    ax.scatter(sub.loc[sub.fn, 'ra'], sub.loc[sub.fn, 'dec'], s=10, c='crimson', label=f'FN={int(sub.fn.sum())}', alpha=0.9)
    ax.scatter(sub.loc[sub.fp, 'ra'], sub.loc[sub.fp, 'dec'], s=8, facecolors='none', edgecolors='cyan',
               linewidths=0.7, label=f'FP={int(sub.fp.sum())}')
    ax.set_title(f'triage @ t*={t:.2f}')
    ax.legend(frameon=False, fontsize=8, loc='best')

    n_s5 = int(sub.y.sum())
    fig.suptitle(
        f"cell {cell_id}  role={role}  N={len(sub):,}  S5={n_s5}  "
        f"mean p={sub.p.mean():.3f}  holdout members="
        f"{int(((sub.y) & sub.stream_label.isin(set(CELL_SPLIT.holdout_streams))).sum())}",
        y=1.02,
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


# Rank discovery cells by S5 content, then by mean predicted score
cell_sum = (
    STARS.groupby(['cell_id', 'role'], as_index=False)
    .agg(n_stars=('ra', 'size'), n_s5=('y', 'sum'), mean_p=('p', 'mean'), n_tp=('tp', 'sum'), n_fp=('fp', 'sum'), n_fn=('fn', 'sum'))
)
disc_cells = cell_sum[cell_sum.role == 'test'].sort_values(['n_s5', 'mean_p'], ascending=False)
train_cells = cell_sum[cell_sum.role.isin(['train', 'val'])].sort_values(['n_s5', 'mean_p'], ascending=False)
print('Top discovery cells by S5 content:')
display(disc_cells.head(12))


### 6a. Discovery cells with known streams (recovered vs missed)


In [ ]:
N_SHOW = 8
for cid in disc_cells.loc[disc_cells.n_s5 > 0, 'cell_id'].head(N_SHOW):
    plot_cell_discovery(int(cid))


### 6b. High-score discovery cells with **no** catalogued S5 (candidate new structure / FP)


In [ ]:
cand = disc_cells[(disc_cells.n_s5 == 0) & (disc_cells.mean_p > 0.15)].sort_values('mean_p', ascending=False)
print(f'{len(cand)} empty discovery cells with mean p > 0.15')
display(cand.head(10))
for cid in cand.cell_id.head(6):
    plot_cell_discovery(int(cid))


### 6c. Supervised cells (sanity — model should recover train streams)


In [ ]:
for cid in train_cells.loc[train_cells.n_s5 > 0, 'cell_id'].head(4):
    plot_cell_discovery(int(cid))


### 6d. Browse any cell


In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display as _display

    _ids = sorted(STARS.cell_id.unique().tolist())
    _dd = W.Dropdown(options=_ids, description='cell_id')
    _out = W.Output()

    def _on_change(change):
        if change['name'] != 'value':
            return
        _out.clear_output(wait=True)
        with _out:
            plot_cell_discovery(int(change['new']))

    _dd.observe(_on_change)
    _display(_dd, _out)
    with _out:
        plot_cell_discovery(int(_dd.value))
except Exception as exc:
    print('ipywidgets unavailable — call plot_cell_discovery(cell_id) manually.', exc)


## 7. Holdout-stream focus

Did fine-tuning on *other* streams transfer to the held-out names in discovery cells?


In [ ]:
focus = STARS[(STARS.role == 'test') & (STARS.stream_label.isin(holdout))]
if not len(focus):
    print('No holdout-stream members land in test cells (they may sit only in train cells that also host train streams).')
    # Show holdout members wherever they are
    focus_all = STARS[STARS.stream_label.isin(holdout)]
    print('Holdout members by cell role:')
    display(focus_all.groupby(['role', 'stream_label']).size().unstack(fill_value=0))
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, g in focus.groupby('stream_label'):
        ax.scatter(g.loc[g.tp, 'ra'], g.loc[g.tp, 'dec'], s=12, label=f'{name} TP', alpha=0.9)
        ax.scatter(g.loc[g.fn, 'ra'], g.loc[g.fn, 'dec'], s=12, marker='x', label=f'{name} FN', alpha=0.9)
    ax.invert_xaxis()
    ax.set_xlabel('RA'); ax.set_ylabel('Dec')
    ax.set_title('Holdout streams in discovery cells: recovered (TP) vs missed (FN)')
    ax.legend(frameon=False, fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()
    display(rec_test[rec_test.is_holdout] if len(rec_test) else rec_test)

# Always show holdout recall split by role
rows = []
for role in ('train', 'val', 'test'):
    tab = stream_recovery_table(STARS, roles=[role])
    for _, r in tab.iterrows():
        if r.stream_label in holdout:
            rows.append({**r.to_dict(), 'role': role})
holdout_by_role = pd.DataFrame(rows)
print('Holdout stream recall by cell role:')
display(holdout_by_role)


## 8. Export

Optional: write the per-star triage table for downstream analysis.


In [ ]:
OUT_DIR = REPO / "notebooks" / "streams" / "figures" / "real_stream_discovery"
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_parquet = OUT_DIR / f"stars_t{T_STAR:.2f}_{RUN_DIR.name[:40]}.parquet"
# Keep a lean export
export_cols = ['cell_id', 'role', 'ra', 'dec', 'p', 'y', 'stream_label', 'pred', 'tp', 'fp', 'fn']
STARS[export_cols].to_parquet(out_parquet, index=False)
print('wrote', out_parquet, 'rows', len(STARS))
meta = {
    'run_dir': str(RUN_DIR),
    'ckpt': str(CKPT),
    't_star': T_STAR,
    'train_cells': sorted(CELL_SPLIT.train_cells),
    'val_cells': sorted(CELL_SPLIT.val_cells),
    'test_cells': sorted(CELL_SPLIT.test_cells),
    'holdout_streams': sorted(CELL_SPLIT.holdout_streams),
    'train_streams': sorted(CELL_SPLIT.train_streams),
}
import json
(OUT_DIR / 'last_discovery_meta.json').write_text(json.dumps(meta, indent=2) + '\n')
print('meta', OUT_DIR / 'last_discovery_meta.json')
